# Training curves

Parses the per-epoch validation lines that training prints, so a captured stdout log can
be turned into curves. Point `LOG_FILES` at one or more logs.

In [ ]:
# Resolve every path through the package config, so this notebook follows whatever
# SPARSEST_DATA_ROOT / SPARSEST_OUTPUT_ROOT point at. Switch datasets here.
CONFIG_NAME = "ipad"   # or "mnist"

import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from sparsest.config import load_config
cfg = load_config(REPO / "configs" / f"{CONFIG_NAME}.yaml")
print("dataset:", cfg.dataset, "| window:", cfg.window)
print("outputs:", cfg.output_path("checkpoints", create=False).parent)

In [ ]:
import re
import matplotlib.pyplot as plt
from collections import defaultdict

FIG_DIR = cfg.output_path("figures")

# Training writes lines like "valid mse for epoch 12: 0.00318."
PATTERNS = {
    "train_stch": re.compile(r"train STCH for epoch (\d+): ([0-9.eE+-]+)"),
    "valid_stch": re.compile(r"valid STCH for epoch (\d+): ([0-9.eE+-]+)"),
    "valid_mse": re.compile(r"valid mse for epoch (\d+): ([0-9.eE+-]+)"),
    "valid_sparsity": re.compile(r"valid l0 sparsity of epoch (\d+): ([0-9.eE+-]+)"),
    "valid_occupancy": re.compile(r"valid l1 occupancy of epoch (\d+): ([0-9.eE+-]+)"),
}


def parse_log(path):
    series = defaultdict(dict)
    with open(path, errors="ignore") as handle:
        # Progress bars use carriage returns; split on them so lines are recoverable.
        for line in handle.read().replace("\r", "\n").splitlines():
            for name, pattern in PATTERNS.items():
                m = pattern.search(line)
                if m:
                    series[name][int(m.group(1))] = float(m.group(2).rstrip("."))
    return {k: (sorted(v), [v[e] for e in sorted(v)]) for k, v in series.items()}

In [ ]:
LOG_FILES = {
    # "w=0.75": "/path/to/run.log",
}

if not LOG_FILES:
    print("Set LOG_FILES to the training logs you captured, e.g.")
    print('  LOG_FILES = {"w=0.75": "runs/ipad/w075.log"}')
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for label, path in LOG_FILES.items():
        s = parse_log(path)
        for ax, key, title in zip(
            axes,
            ["valid_mse", "valid_occupancy", "valid_stch"],
            ["Validation MSE", "Validation occupancy", "Validation STCH"],
        ):
            if key in s:
                ax.plot(*s[key], marker="o", markersize=3, label=label)
                ax.set_title(title); ax.set_xlabel("Epoch")
                ax.grid(True, linestyle="--", alpha=0.4)
    axes[0].legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"training_curves_{CONFIG_NAME}.pdf")